In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')
print('Train shape:', train.shape)
print('Test shape:', test.shape)
train.head()

Train shape: (891, 12)
Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [4]:
train.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [5]:
train.drop(columns=['Cabin', 'Name', 'Ticket', 'PassengerId'], inplace=True)

In [6]:
train['Age'] = train['Age'].fillna(train['Age'].median())
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])
train.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,0


In [7]:
train['family_size'] = train['SibSp'] + train['Parch'] + 1
train['is_alone'] = (train['family_size']==1).astype(int)
train['fare_per_person'] = train['Fare']/train['family_size']

In [8]:
le = LabelEncoder()
train['Sex'] = le.fit_transform(train['Sex'])

dummies = pd.get_dummies(train['Embarked'], prefix='Embarked', drop_first=True)
train = pd.concat([train, dummies], axis=1)
train.drop(columns='Embarked', inplace=True)

In [9]:
X = train.drop('Survived', axis=1)
y = train['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Accuracy score:', accuracy_score(y_test, y_pred))

Accuracy score: 0.8156424581005587


In [11]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10]
}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, verbose=1)
grid.fit(X_train, y_train)
print('Best params:', grid.best_params_)
print('Best Accuracy:', grid.best_score_)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best params: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 300}
Best Accuracy: 0.8244164286417808


In [12]:
test['Age'] = test['Age'].fillna(test['Age'].median())
test['Fare'] = test['Fare'].fillna(test['Fare'].median())
test['Embarked'] = test['Embarked'].fillna(test['Embarked'].mode()[0])
test.drop(columns=['Cabin', 'Name', 'Ticket'], inplace=True)


In [14]:
test['family_size'] = test['SibSp'] + test['Parch'] + 1
test['is_alone'] = (test['family_size']==1).astype(int)
test['fare_per_person'] = test['Fare']/test['family_size']

In [15]:
test['Sex'] = le.transform(test['Sex'])
dummies = pd.get_dummies(test['Embarked'], prefix='Embarked', drop_first=True)
test = pd.concat([test, dummies], axis=1)
test.drop(columns='Embarked', inplace=True)

In [17]:
X_test_Kaggle = test.drop('PassengerId', axis=1)
predictions = grid.predict(X_test_Kaggle)
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})
submission.to_csv('submission.csv', index=False)
print('Submission file ready')
print(submission.head())

Submission file ready
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         0
